<a href="https://colab.research.google.com/github/ngoanlc25ai/AdvanceDataScience/blob/main/Lung-EffNet-Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#About The Project
This project focuses on lung cancer detection using CT scan images from the IQ-OTH/NCCD Lung Cancer Dataset, collected by The Iraq Oncology Teaching Hospital / National Center for Cancer Diseases. The dataset includes three classes: Normal, Benign, and Malignant.

A ResNet-18 model pretrained on ImageNet was used through transfer learning. The final fully connected layer was replaced to match the three output classes, and all layers were fine-tuned to adapt the model to medical imaging data. The model was trained using CrossEntropyLoss, the AdamW optimizer, and a ReduceLROnPlateau learning rate scheduler.

Performance was evaluated using test loss, accuracy, classification report, and a confusion matrix. The fine-tuned ResNet-18 achieved strong classification performance, demonstrating the effectiveness of transfer learning for automated lung cancer detection.

#Re-Check Images

In [ ]:
import kagglehub
from pathlib import Path
from collections import defaultdict
import json
from PIL import Image
import numpy as np

# Download dataset
path = kagglehub.dataset_download("adityamahimkar/iqothnccd-lung-cancer-dataset")
print("Path to dataset files:", path)

# ── Tự động tìm ảnh trong path vừa download ──
root = Path(path)
ALLOWED_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}
EXPECTED_MODES = {"RGB", "L"}

image_files = [f for f in root.rglob("*")
               if f.is_file() and f.suffix.lower() in ALLOWED_EXTS]

print(f"\n📁 Dataset: {root}")
print(f"🖼️  Tổng số ảnh: {len(image_files)}")

if len(image_files) == 0:
    all_files = list(root.rglob("*.*"))
    exts = set(f.suffix.lower() for f in all_files if f.is_file())
    print(f"⚠️  Không tìm thấy ảnh! Extensions thực tế: {exts}")
    for f in all_files[:10]:
        print(f"  {f.relative_to(root)}")
else:
    errors        = []
    size_counter  = defaultdict(int)
    mode_counter  = defaultdict(int)
    class_counter = defaultdict(int)
    ext_counter   = defaultdict(int)

    for i, p in enumerate(image_files, 1):
        if i % 300 == 0:
            print(f"  ... {i}/{len(image_files)}")
        issues = []
        class_counter[p.parent.name] += 1
        ext_counter[p.suffix.lower()] += 1

        if p.stat().st_size < 1024:
            issues.append(f"File quá nhỏ ({p.stat().st_size} bytes)")

        try:
            with Image.open(p) as img:
                w, h, mode = img.width, img.height, img.mode
                size_counter[f"{w}x{h}"] += 1
                mode_counter[mode] += 1
                if mode not in EXPECTED_MODES:
                    issues.append(f"Mode không hợp lệ: {mode}")
                if np.array(img).std() < 1.0:
                    issues.append("Ảnh một màu (std≈0)")
        except Exception as e:
            issues.append(f"Không đọc được: {e}")

        if issues:
            errors.append({"file": str(p.relative_to(root)), "issues": issues})

    total = len(image_files)
    n_ok  = total - len(errors)

    print(f"\n{'═'*55}")
    print(f"  Tổng ảnh    : {total}")
    print(f"  ✅ Đạt chuẩn : {n_ok}  ({n_ok/total*100:.1f}%)")
    print(f"  ❌ Có vấn đề : {len(errors)}  ({len(errors)/total*100:.1f}%)")

    print(f"\n  📂 PHÂN BỐ THEO CLASS:")
    max_c = max(class_counter.values())
    for cls, cnt in sorted(class_counter.items()):
        bar = "█" * int(cnt / max_c * 25)
        print(f"    {cls:<30} {cnt:>4}  {bar}")

    print(f"\n  📐 KÍCH THƯỚC PHỔ BIẾN:")
    for size, cnt in sorted(size_counter.items(), key=lambda x: -x[1])[:8]:
        print(f"    {size:<15} {cnt:>4} ảnh")

    print(f"\n  🎨 MODE MÀU:")
    for mode, cnt in sorted(mode_counter.items(), key=lambda x: -x[1]):
        mark = "✅" if mode in EXPECTED_MODES else "❌"
        print(f"    {mode:<10} {cnt:>4} ảnh  {mark}")

    print(f"\n  📄 ĐỊNH DẠNG FILE:")
    for ext, cnt in sorted(ext_counter.items(), key=lambda x: -x[1]):
        print(f"    {ext:<10} {cnt:>4} ảnh")

    if errors:
        print(f"\n  ❌ MẪU LỖI (tối đa 20):")
        for e in errors[:20]:
            print(f"  [{e['file']}]")
            for iss in e["issues"]:
                print(f"    → {iss}")
        out = Path("/kaggle/working/error_images.json")
        out.write_text(json.dumps(errors, ensure_ascii=False, indent=2))
        print(f"\n  📋 Đã lưu lỗi → {out}")
    else:
        print(f"\n  🎉 Tất cả ảnh đều đạt chuẩn!")
    print(f"{'═'*55}")

# Comperation
Model Params Size
*   Inception-ResNet-V2 55M 215MB,
*   EfficientNet-B1 7.8M 31MB
*   ResNet-50 25M 98MB

In [ ]:
"""
Lung Cancer Classification - 3 Model Comparison
Models: Inception-ResNet-V2 | EfficientNet-B1 | ResNet-50
Framework: PyTorch + torchvision + timm
"""

# ─── 0. INSTALL & IMPORTS ──────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "timm", "-q"], check=True)

import os, time, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import (
    resnet50,        ResNet50_Weights,
    efficientnet_b1, EfficientNet_B1_Weights,
)
import timm  # Inception-ResNet-V2
from sklearn.metrics import classification_report, confusion_matrix

# ─── 1. CONFIG ─────────────────────────────────────────────────────────────────
import kagglehub
path     = kagglehub.dataset_download("adityamahimkar/iqothnccd-lung-cancer-dataset")
DATA_DIR = path
OUT_DIR  = "/kaggle/working/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE     = 299   # Inception-ResNet-V2 yêu cầu 299x299; các model khác vẫn dùng được
BATCH_SIZE   = 16    # giảm xuống vì Inception-ResNet-V2 lớn hơn
EPOCHS       = 15
LR           = 1e-4
WEIGHT_DECAY = 1e-4
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ─── 2. DATASET ────────────────────────────────────────────────────────────────
# Inception-ResNet-V2 dùng mean/std của ImageNet (giống các model khác)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Tự động detect subfolder chứa ảnh
root = Path(DATA_DIR)
candidates = [root] + [d for d in root.iterdir() if d.is_dir()]
DATA_ROOT = next(
    (d for d in candidates
     if any(True for _ in d.glob("*/*.jpg")) or
        any(True for _ in d.glob("*/*.png"))),
    root
)
print(f"Data root: {DATA_ROOT}")

full_ds     = datasets.ImageFolder(str(DATA_ROOT))
classes     = full_ds.classes
NUM_CLASSES = len(classes)
print(f"Classes ({NUM_CLASSES}): {classes}")
print(f"Total samples: {len(full_ds)}")

# Train / Val / Test split 70/15/15
n       = len(full_ds)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_ds.dataset.transform = train_tf
val_ds.dataset.transform   = val_tf
test_ds.dataset.transform  = val_tf

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# ─── 3. MODEL FACTORY ──────────────────────────────────────────────────────────
def build_model(name: str) -> nn.Module:
    if name == "resnet50":
        m = resnet50(weights=ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)

    elif name == "efficientnet_b1":
        m = efficientnet_b1(weights=EfficientNet_B1_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)

    elif name == "inception_resnet_v2":
        # timm: inception_resnet_v2 pretrained trên ImageNet
        m = timm.create_model("inception_resnet_v2", pretrained=True, num_classes=NUM_CLASSES)

    return m.to(DEVICE)


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def model_size_mb(fpath: str) -> float:
    return os.path.getsize(fpath) / (1024 ** 2)

# ─── 4. TRAIN / EVAL FUNCTIONS ────────────────────────────────────────────────
def trainOneEpoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct = 0, 0
    for imgs, labels in tqdm(loader, desc="Train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
    return running_loss / len(loader.dataset), correct / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct = 0, 0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Eval", leave=True):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            correct += (preds == labels).sum().item()
    return (
        running_loss / len(loader.dataset),
        correct / len(loader.dataset),
        np.array(preds_all),
        np.array(labels_all),
    )

# ─── 5. TRAINING LOOP ─────────────────────────────────────────────────────────
MODEL_CONFIGS = [
    ("inception_resnet_v2", "Inception-ResNet-V2"),
    ("efficientnet_b1",     "EfficientNet-B1"),
    ("resnet50",            "ResNet-50"),
]

all_results = {}

for model_key, model_name in MODEL_CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"{'='*60}")

    model     = build_model(model_key)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=3)

    weight_path   = f"{OUT_DIR}/{model_key}_weights.pth"
    best_val_loss = np.inf
    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []
    t0 = time.time()

    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        train_loss, train_acc         = trainOneEpoch(model, train_dl, criterion, optimizer)
        val_loss,   val_acc, _, _     = evaluate(model, val_dl, criterion)
        scheduler.step(val_loss)

        train_losses.append(train_loss); val_losses.append(val_loss)
        train_accs.append(train_acc);    val_accs.append(val_acc)

        print(f"  Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f}")
        print(f"  Val   Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), weight_path)
            print("  ✅ Model saved")

    train_time = time.time() - t0

    # ── Evaluate on test set ──
    model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
    test_loss, test_acc, preds, labels = evaluate(model, test_dl, criterion)
    print(f"\nTest Loss: {test_loss:.4f}  Test Acc: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(labels, preds, target_names=classes))

    # ── Confusion Matrix ──
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes)
    plt.title(f"Confusion Matrix — {model_name}", fontsize=14)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{model_key}_confusion.png", dpi=120)
    plt.show()

    # ── Training Curve ──
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses,   label="Val Loss")
    plt.title(f"Training Curve — {model_name}", fontsize=14)
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{model_key}_curve.png", dpi=120)
    plt.show()

    # ── Lưu kết quả ──
    all_results[model_key] = {
        "name":         model_name,
        "test_acc":     round(test_acc * 100, 2),
        "test_loss":    round(test_loss, 4),
        "best_val_acc": round(max(val_accs) * 100, 2),
        "params_M":     round(count_params(model) / 1e6, 2),
        "size_MB":      round(model_size_mb(weight_path), 2),
        "train_time_s": round(train_time, 1),
        "train_losses": train_losses,
        "val_losses":   val_losses,
        "train_accs":   train_accs,
        "val_accs":     val_accs,
    }

    # Giải phóng VRAM trước khi train model tiếp theo
    del model
    torch.cuda.empty_cache()

# ─── 6. COMPARISON DASHBOARD ──────────────────────────────────────────────────
print("\n" + "="*60)
print("  MODEL COMPARISON DASHBOARD")
print("="*60)

names     = [v["name"]         for v in all_results.values()]
test_accs = [v["test_acc"]     for v in all_results.values()]
params    = [v["params_M"]     for v in all_results.values()]
sizes     = [v["size_MB"]      for v in all_results.values()]
times     = [v["train_time_s"] for v in all_results.values()]
colors    = ["#4C72B0", "#DD8452", "#55A868"]

fig = plt.figure(figsize=(18, 14))
fig.suptitle("Model Comparison — Lung Cancer Classification\n"
             "Inception-ResNet-V2  |  EfficientNet-B1  |  ResNet-50",
             fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

# ── 6a. Test Accuracy ──
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(names, test_accs, color=colors, edgecolor="white", linewidth=1.5)
ax1.set_title("Test Accuracy (%)", fontweight="bold")
ax1.set_ylim(0, 115)
ax1.set_ylabel("%")
for bar, val in zip(bars, test_accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{val:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=10)
ax1.tick_params(axis="x", rotation=20)

# ── 6b. Parameters ──
ax2 = fig.add_subplot(gs[0, 1])
bars2 = ax2.bar(names, params, color=colors, edgecolor="white", linewidth=1.5)
ax2.set_title("Parameters (M)", fontweight="bold")
ax2.set_ylabel("Millions")
for bar, val in zip(bars2, params):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{val}M", ha="center", va="bottom", fontweight="bold", fontsize=10)
ax2.tick_params(axis="x", rotation=20)

# ── 6c. Model Size ──
ax3 = fig.add_subplot(gs[0, 2])
bars3 = ax3.bar(names, sizes, color=colors, edgecolor="white", linewidth=1.5)
ax3.set_title("Model Size (MB)", fontweight="bold")
ax3.set_ylabel("MB")
for bar, val in zip(bars3, sizes):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{val} MB", ha="center", va="bottom", fontweight="bold", fontsize=10)
ax3.tick_params(axis="x", rotation=20)

# ── 6d. Loss Curves overlay ──
ax4 = fig.add_subplot(gs[1, :2])
for (key, res), color in zip(all_results.items(), colors):
    ax4.plot(res["train_losses"], label=f"{res['name']} Train", color=color, linewidth=2)
    ax4.plot(res["val_losses"],   label=f"{res['name']} Val",   color=color, linewidth=2, linestyle="--")
ax4.set_title("Loss Curves — All Models", fontweight="bold")
ax4.set_xlabel("Epoch"); ax4.set_ylabel("Loss")
ax4.legend(fontsize=7); ax4.grid(alpha=0.3)

# ── 6e. Val Accuracy curves overlay ──
ax5 = fig.add_subplot(gs[1, 2])
for (key, res), color in zip(all_results.items(), colors):
    ax5.plot([a * 100 for a in res["val_accs"]], label=res["name"], color=color, linewidth=2)
ax5.set_title("Val Accuracy (%)", fontweight="bold")
ax5.set_xlabel("Epoch"); ax5.set_ylabel("%")
ax5.legend(fontsize=7); ax5.grid(alpha=0.3)

# ── 6f. Scatter: Accuracy vs Params ──
ax6 = fig.add_subplot(gs[2, :2])
for (key, res), color in zip(all_results.items(), colors):
    ax6.scatter(res["params_M"], res["test_acc"],
                color=color, s=max(res["size_MB"] * 8, 80),
                label=res["name"], zorder=5, edgecolors="white", linewidth=1.5)
    ax6.annotate(res["name"],
                 (res["params_M"], res["test_acc"]),
                 textcoords="offset points", xytext=(8, 4), fontsize=9)
ax6.set_title("Accuracy vs Parameters  (bubble size ∝ model size)", fontweight="bold")
ax6.set_xlabel("Parameters (M)"); ax6.set_ylabel("Test Accuracy (%)")
ax6.grid(alpha=0.3); ax6.legend()

# ── 6g. Summary Table ──
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis("off")
table_data = [["Model", "Acc %", "Params", "Size", "Time"]]
for v in all_results.values():
    table_data.append([
        v["name"],
        f"{v['test_acc']}%",
        f"{v['params_M']}M",
        f"{v['size_MB']}MB",
        f"{v['train_time_s']:.0f}s",
    ])
tbl = ax7.table(cellText=table_data[1:], colLabels=table_data[0],
                cellLoc="center", loc="center",
                bbox=[0, 0.05, 1, 0.88])
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)

for j in range(5):
    tbl[0, j].set_facecolor("#2C3E50")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

row_colors = ["#EBF5FB", "#FEF9E7", "#EAFAF1"]
for i, rc in enumerate(row_colors, 1):
    for j in range(5):
        tbl[i, j].set_facecolor(rc)

best_idx = test_accs.index(max(test_accs)) + 1
for j in range(5):
    tbl[best_idx, j].set_text_props(fontweight="bold")

ax7.set_title("Summary Table\n(⭐ = best accuracy)", fontweight="bold", pad=10)

plt.savefig(f"{OUT_DIR}/comparison_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\n📊 Dashboard saved → {OUT_DIR}/comparison_dashboard.png")

# ── 6h. Text Summary ──
print(f"\n{'─'*62}")
print(f"  {'Model':<24} {'Acc':>7} {'Params':>9} {'Size':>8} {'Time':>8}")
print(f"{'─'*62}")
for v in all_results.values():
    star = " ⭐" if v["test_acc"] == max(test_accs) else ""
    print(f"  {v['name']:<24} {v['test_acc']:>6.1f}% "
          f"{v['params_M']:>7.2f}M "
          f"{v['size_MB']:>6.1f}MB "
          f"{v['train_time_s']:>6.0f}s{star}")
print(f"{'─'*62}")

# Lưu JSON
summary = {k: {kk: vv for kk, vv in v.items()
               if kk not in ("train_losses", "val_losses", "train_accs", "val_accs")}
           for k, v in all_results.items()}
with open(f"{OUT_DIR}/results.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"📋 Results saved → {OUT_DIR}/results.json")